# 12. Comparación de CPU, GPU y Metal para Deep Learning

Este notebook compara el rendimiento de entrenamiento de modelos en diferentes dispositivos, con benchmarks y visualizaciones.

## Objetivo
- Entender las diferencias entre CPU, GPU (CUDA) y Apple Metal para deep learning.
- Medir y comparar tiempos de entrenamiento.
- Identificar cuándo vale la pena usar GPU.
- Aplicar buenas prácticas de configuración de hardware.

## Prerequisitos

> 📌 **Prerequisitos:** Haber completado los notebooks [04 (MLP)](./04_redes_neuronales_capa_densa.ipynb) y [05 (CNN)](./05_redes_convolucionales_cnn.ipynb).

- Conceptos de redes neuronales y entrenamiento con TensorFlow/Keras.

## 1. Introducción teórica

| Dispositivo | Ventaja | Desventaja | Cuándo usar |
|-------------|---------|------------|-------------|
| **CPU** | Universal, fácil de configurar | Lento para modelos grandes | Modelos pequeños, preprocesamiento |
| **GPU (CUDA)** | Masivamente paralela | Requiere NVIDIA + drivers | Modelos medianos/grandes |
| **Apple Metal** | Integrada en Mac M1/M2/M3 | Soporte limitado | Desarrollo en Mac |

## 2. Configuración del entorno

In [ ]:
import numpy as np
import tensorflow as tf
import time
import matplotlib.pyplot as plt
import platform

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

print('TensorFlow:', tf.__version__)
print('Dispositivos disponibles:', tf.config.list_physical_devices())
print('Sistema operativo:', platform.system())
print('Procesador:', platform.processor())

## 3. Preparar datos y modelo

In [ ]:
# Cargar MNIST
mnist = tf.keras.datasets.mnist
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train, X_test = X_train / 255.0, X_test / 255.0

def create_mlp():
    """Modelo MLP simple para benchmark."""
    model = tf.keras.Sequential([
        tf.keras.layers.Flatten(input_shape=(28, 28)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def create_cnn():
    """Modelo CNN más complejo para notar la diferencia CPU/GPU."""
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2,2)),
        tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2,2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Datos para CNN
X_train_cnn = np.expand_dims(X_train, -1)
X_test_cnn = np.expand_dims(X_test, -1)

## 4. Benchmark en CPU

In [ ]:
# Benchmark MLP en CPU
with tf.device('/CPU:0'):
    model_cpu = create_mlp()
    start = time.time()
    model_cpu.fit(X_train, y_train, epochs=3, batch_size=128, validation_split=0.1, verbose=2)
    cpu_time_mlp = time.time() - start

# Benchmark CNN en CPU
with tf.device('/CPU:0'):
    model_cpu_cnn = create_cnn()
    start = time.time()
    model_cpu_cnn.fit(X_train_cnn, y_train, epochs=3, batch_size=128, validation_split=0.1, verbose=2)
    cpu_time_cnn = time.time() - start

print(f"\nTiempo MLP en CPU: {cpu_time_mlp:.2f}s")
print(f"Tiempo CNN en CPU: {cpu_time_cnn:.2f}s")

## 5. Benchmark en GPU (si disponible)

In [ ]:
gpu_devices = tf.config.list_physical_devices('GPU')
gpu_time_mlp = None
gpu_time_cnn = None

if gpu_devices:
    print(f'GPU detectada: {gpu_devices}')
    with tf.device('/GPU:0'):
        # MLP
        model_gpu = create_mlp()
        start = time.time()
        model_gpu.fit(X_train, y_train, epochs=3, batch_size=128, validation_split=0.1, verbose=2)
        gpu_time_mlp = time.time() - start
        # CNN
        model_gpu_cnn = create_cnn()
        start = time.time()
        model_gpu_cnn.fit(X_train_cnn, y_train, epochs=3, batch_size=128, validation_split=0.1, verbose=2)
        gpu_time_cnn = time.time() - start
    print(f"\nTiempo MLP en GPU: {gpu_time_mlp:.2f}s")
    print(f"Tiempo CNN en GPU: {gpu_time_cnn:.2f}s")
else:
    print("No se detectó GPU disponible.")

## 6. Benchmark en Metal (Apple Silicon, si disponible)

In [ ]:
metal_time_mlp = None
metal_time_cnn = None

if platform.system() == 'Darwin' and gpu_devices:
    try:
        with tf.device('/GPU:0'):
            model_metal = create_mlp()
            start = time.time()
            model_metal.fit(X_train, y_train, epochs=3, batch_size=128, validation_split=0.1, verbose=2)
            metal_time_mlp = time.time() - start
            model_metal_cnn = create_cnn()
            start = time.time()
            model_metal_cnn.fit(X_train_cnn, y_train, epochs=3, batch_size=128, validation_split=0.1, verbose=2)
            metal_time_cnn = time.time() - start
        print(f"Tiempo MLP en Metal: {metal_time_mlp:.2f}s")
        print(f"Tiempo CNN en Metal: {metal_time_cnn:.2f}s")
    except Exception as e:
        print("No se pudo entrenar usando Metal:", e)
else:
    print("Metal solo disponible en macOS con Apple Silicon.")

## 7. Comparación de resultados

In [ ]:
import pandas as pd

results = []
results.append({'Dispositivo': 'CPU', 'MLP (s)': cpu_time_mlp, 'CNN (s)': cpu_time_cnn})
if gpu_time_mlp is not None:
    results.append({'Dispositivo': 'GPU', 'MLP (s)': gpu_time_mlp, 'CNN (s)': gpu_time_cnn})
if metal_time_mlp is not None:
    results.append({'Dispositivo': 'Metal', 'MLP (s)': metal_time_mlp, 'CNN (s)': metal_time_cnn})

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['gray', 'green', 'blue'][:len(df_results)]

axes[0].bar(df_results['Dispositivo'], df_results['MLP (s)'], color=colors)
axes[0].set_ylabel('Tiempo (s)')
axes[0].set_title('Benchmark MLP')

axes[1].bar(df_results['Dispositivo'], df_results['CNN (s)'], color=colors)
axes[1].set_ylabel('Tiempo (s)')
axes[1].set_title('Benchmark CNN')

plt.tight_layout()
plt.show()

## 8. Discusión y Conclusiones

- La GPU muestra mayor ventaja con modelos más complejos (CNN vs MLP).
- Para modelos pequeños, la CPU puede ser comparable o incluso más rápida.
- La diferencia se amplifica con más datos y más épocas.
- Verifica compatibilidad de hardware y drivers antes de depender de GPU.

## 9. Ejercicios Propuestos

1. **Ejercicio 1:** Aumenta el número de epochs a 10 y compara. ¿Se amplifica la diferencia?

2. **Ejercicio 2:** Varía el `batch_size` (32, 128, 512) y mide tiempos. ¿Cuál es la configuración óptima en GPU?

3. **Ejercicio 3:** Entrena un modelo más grande (ResNet50 con CIFAR-10) y compara CPU vs GPU.

4. **Ejercicio 4 (Avanzado):** Usa `tf.data.Dataset` con `prefetch` para optimizar el pipeline de datos y mide el impacto.

## 10. Referencias y Recursos

- [TensorFlow GPU Guide](https://www.tensorflow.org/guide/gpu)
- [Apple Metal TensorFlow Plugin](https://developer.apple.com/metal/tensorflow-plugin/)
- [Working with GPUs - Keras](https://keras.io/guides/working_with_gpus/)

---

📎 **Notebook anterior:** [11. Interpretabilidad de Modelos](./11_interpretabilidad_modelos.ipynb)  
📎 **Notebook siguiente:** [13. Despliegue de Modelos](./13_despliegue_modelos.ipynb)